# Comprehension-Time Study — Statistical Analysis

Run this notebook on the CSV exported from the admin dashboard
(**Export & backup** → ⬇️ Full responses CSV).

Produces all the tables and statistics you need for the thesis chapter:
- Descriptive table (Language × Condition)
- Speedup percentages vs Jain et al. (2024) 31.9%
- Paired t-tests per language
- Per-model breakdown (Gemini vs Qwen)
- Accuracy table

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Load data

In [ ]:
# Update path to wherever you saved the export
df = pd.read_csv('tta_responses.csv')

df['tta_seconds'] = pd.to_numeric(df['tta_seconds'], errors='coerce')
df['sample_id'] = pd.to_numeric(df['sample_id'], errors='coerce')

# Auto-flag accuracy
def is_correct(g, gold):
    if pd.isna(g) or pd.isna(gold): return False
    g, gold = str(g).strip().lower(), str(gold).strip().lower()
    if not g or not gold: return False
    return gold in g or g in gold

df['correct'] = df.apply(lambda r: is_correct(r['given_answer'], r['gold_answer']), axis=1)

print(f'Loaded {len(df)} responses from {df.participant.nunique()} participants')
print(f'Languages: {sorted(df.language.unique())}')
print(f'Conditions: {sorted(df.condition.unique())}')
df.head()

## 2. Descriptive table — Mean TTA per (Language × Condition)

In [ ]:
desc = df.groupby(['language', 'condition'])['tta_seconds'].agg(
    n='count', mean='mean', std='std', median='median'
).round(2).reset_index()
print(desc)

## 3. Speedup table — compare to Jain et al. (2024) 31.9%

In [ ]:
pivot = df.groupby(['language', 'condition'])['tta_seconds'].mean().unstack().round(2)

if 'passage' in pivot.columns and 'mindmap' in pivot.columns:
    pivot['mindmap_vs_text_speedup_%'] = (
        (pivot['passage'] - pivot['mindmap']) / pivot['passage'] * 100
    ).round(1)
if 'passage' in pivot.columns and 'both' in pivot.columns:
    pivot['both_vs_text_speedup_%'] = (
        (pivot['passage'] - pivot['both']) / pivot['passage'] * 100
    ).round(1)

print(pivot)
print(f"\nReference: Jain et al. (2024) reported 31.9% speedup for English mind maps.")

## 4. Paired t-tests per language

For each language, each (sample_id, qid) is matched across the two conditions
being compared. The paired t-test operates on these N matched questions.

In [ ]:
results = []
pairs = [('passage', 'mindmap'), ('passage', 'both'), ('mindmap', 'both')]

for lang in sorted(df.language.unique()):
    sub = df[df.language == lang]
    for c1, c2 in pairs:
        d1 = sub[sub.condition == c1].groupby(['sample_id', 'qid'])['tta_seconds'].mean()
        d2 = sub[sub.condition == c2].groupby(['sample_id', 'qid'])['tta_seconds'].mean()
        # Match on the same (sample_id, qid)
        common = d1.index.intersection(d2.index)
        x1 = d1.loc[common].values
        x2 = d2.loc[common].values
        if len(x1) < 3:
            continue
        t, p = stats.ttest_rel(x1, x2)
        speedup = (x1.mean() - x2.mean()) / x1.mean() * 100
        # Cohen's d for paired samples
        diff = x1 - x2
        d_cohen = diff.mean() / diff.std(ddof=1)
        results.append({
            'language': lang, 'comparison': f'{c1} vs {c2}',
            'n_pairs': len(x1),
            'mean_1': round(x1.mean(), 2),
            'mean_2': round(x2.mean(), 2),
            'diff': round(x1.mean() - x2.mean(), 2),
            'speedup_%': round(speedup, 1),
            't': round(t, 3),
            'p': round(p, 5),
            'cohen_d': round(d_cohen, 2),
            'sig': '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        })

ttest_df = pd.DataFrame(results)
print(ttest_df)

## 5. Per-model breakdown — Gemini vs Qwen (mindmap & both only)

In [ ]:
mm = df[df.condition.isin(['mindmap', 'both'])]
model_table = mm.groupby(['language', 'condition', 'model'])['tta_seconds'].agg(
    n='count', mean='mean', std='std'
).round(2).reset_index()
print(model_table)

## 6. Accuracy table

In [ ]:
acc = df.groupby(['language', 'condition'])['correct'].agg(
    n='count', accuracy='mean'
).reset_index()
acc['accuracy_%'] = (acc['accuracy'] * 100).round(1)
print(acc[['language', 'condition', 'n', 'accuracy_%']])

## 7. Mixed-effects model (advanced — robustness check)

If you have `statsmodels` installed, this fits a linear mixed-effects model
accounting for participant and item random effects.

In [ ]:
try:
    import statsmodels.formula.api as smf
    # Per-language model
    for lang in sorted(df.language.unique()):
        sub = df[df.language == lang].copy()
        sub['condition'] = pd.Categorical(sub['condition'],
                                            categories=['passage', 'mindmap', 'both'])
        try:
            model = smf.mixedlm('tta_seconds ~ C(condition)',
                                 data=sub,
                                 groups=sub['sample_id']).fit(reml=False)
            print(f'\n=== {lang.upper()} ===')
            print(model.summary().tables[1])
        except Exception as e:
            print(f'{lang}: {e}')
except ImportError:
    print('Install statsmodels: pip install statsmodels')

## 8. Export final tables for thesis

In [ ]:
# Save all tables as CSVs ready to paste into thesis
desc.to_csv('table_descriptive.csv', index=False)
pivot.to_csv('table_speedup.csv')
ttest_df.to_csv('table_ttests.csv', index=False)
model_table.to_csv('table_per_model.csv', index=False)
acc.to_csv('table_accuracy.csv', index=False)
print('Saved 5 CSV tables for the thesis chapter.')